# Data Pipeline for Senate LDA Expenses
---

This is a data pipe designed to be used to pull out and sum all Senate LDA expense data while removing duplicates created by amendments or double-filling. The data at the end is very similar to Open secrets data, though the Open Secrets lobbying page shows totals that include the income statements from lobbying firms and I don't know much about those so I've left them out to be added at some future date. 

## Input Requirements
- To run this pipeline you will need to first get a csv from lobbyfinder.py
- This pipeline is designed to use all of the inbuild column names from lobbyfinder.py so I don't think it will work with anything else

## To Use
- Make sure all requirements are installed, as I installed the ones I needed as I went
- import your .csv into this folder and change the pdf.readcsv to your .csv name


## Output
- This will leave you with 4 graphics in the Viz folder all built with ploty so that you can interact with them in the notebook to check values and dates. 

### Libraries Used
- pandas, plotly, kaledio, nbformat

### DataFrames in this Pipeline

| DataFrame | Created From | Purpose |
|---|---|---|
| `df` | CSV file | Raw data |
| `df_clean` | `df` | Filtered to `role == 'registrant'` |
| `dupes` | `df_clean` | Diagnostic view — shows quarters with more than one filing |
| `df_deduped` | `df_clean` | Primary clean df — duplicates removed, keeping latest `dt_posted` per client/year/quarter |
| `df_yearly` | `df_deduped` | Expenses summed by company + year — feeds per-company line chart |
| `df_pct` | `df_yearly` | Adds year-over-year % change per company — feeds % change line chart |
| `df_total` | `df_deduped` | All companies

---
### Last Update: 04/26/26
---

## Step 1: dependances and importing the data

In [23]:
%pip install plotly

import sys
!{sys.executable} -m pip install kaleido

import plotly.io as pio
pio.kaleido.scope.mathjax = None



Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\craig\AppData\Local\Temp\ipykernel_14424\1501935392.py:7: DeprecationWarning: 
Use of plotly.io.kaleido.scope.mathjax is deprecated and support will be removed after September 2025.
Please use plotly.io.defaults.mathjax instead.

  pio.kaleido.scope.mathjax = None


In [24]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [25]:
df = pd.read_csv('lobbying_filings_Primes.csv')
df.head()

,source,query_company,role,filing_uuid,filing_year,period,filing_type,dt_posted,registrant_name,registrant_id,client_name,client_id,income,expenses,lobbying_issues,lobbying_bills,lobbying_gov_entities,url
0,Senate LDA,Lockheed Martin,client,49117f1a-a236-4583-a60e-919cc59e79c4,1999,NaN,Year-End Report,2000-02-14T00:00:00-05:00,THE RHOADS GROUP,6644,LOCKHEED MARTIN CORP,108908,120000.0,NaN,Defense; Telecommunications,NaN,HOUSE OF REPRESENTATIVES; SENATE,https://lda.senate.gov/filings/public/filing/4...
1,Senate LDA,Lockheed Martin,client,31f28e72-7236-4837-b0b6-8d262d1d6cd8,1999,NaN,Year-End Report,2000-02-14T00:00:00-05:00,VAN SCOYOC ASSOCIATES,39837,LOCKHEED MARTIN MISSILES & SPACE CO,147700,20000.0,NaN,Defense; Budget/Appropriations,NaN,HOUSE OF REPRESENTATIVES; SENATE,https://lda.senate.gov/filings/public/filing/3...
2,Senate LDA,Lockheed Martin,client,5d60eeb6-241e-4977-b0d1-852a36436cbf,1999,NaN,Mid-Year Termination (No Activity),1999-08-13T00:00:00-04:00,"CARTER GROUP, LLC",49008,LOCKHEED MARTIN AIRCRAFT & LOGISTICS CENTERS,156072,NaN,NaN,NaN,NaN,NaN,https://lda.senate.gov/filings/public/filing/5...
3,Senate LDA,Lockheed Martin,client,d52d60e0-0536-4207-8d87-9c1084e31025,1999,NaN,Year-End Report,2000-02-14T00:00:00-05:00,O'MELVENY & MYERS LLP,29904,LOCKHEED MARTIN CORP,135265,100000.0,NaN,Aerospace,NaN,HOUSE OF REPRESENTATIVES; SENATE,https://lda.senate.gov/filings/public/filing/d...
4,Senate LDA,Lockheed Martin,client,4e6ca9fd-7bad-451e-8a13-d7a698af4815,1999,NaN,Year-End Report,2000-02-14T00:00:00-05:00,DAP & ASSOC,48986,LOCKHEED MARTIN CORP,156067,120000.0,NaN,Telecommunications; Communications/Broadcastin...,NaN,HOUSE OF REPRESENTATIVES; SENATE,https://lda.senate.gov/filings/public/filing/4...


## Step 2 - Data Cleaning

In [26]:
# Filter for [role] = registrant 

df_clean = df[df['role'] == 'registrant']


In [27]:
# Gather Duplicate Quarter reports & Amendments

dupes = df_clean[df_clean.duplicated(subset=['client_name', 'filing_year', 'filing_type'], keep=False)]
dupes[['client_name', 'filing_year', 'filing_type', 'dt_posted']].sort_values(['client_name', 'filing_year', 'filing_type'])


,client_name,filing_year,filing_type,dt_posted
4724,BEN FRANKLIN TRANSIT,2012,2nd Quarter - Report,2012-07-19T14:20:39.710000-04:00
4725,BEN FRANKLIN TRANSIT,2012,2nd Quarter - Report,2012-07-19T14:24:52.830000-04:00
5024,BEN FRANKLIN TRANSIT,2014,4th Quarter - Amendment,2016-01-28T15:40:52.397000-05:00
5026,BEN FRANKLIN TRANSIT,2014,4th Quarter - Amendment,2016-01-28T15:45:02.127000-05:00
5027,BEN FRANKLIN TRANSIT,2014,4th Quarter - Amendment,2016-01-28T15:53:04.280000-05:00
...,...,...,...,...
5506,THERMO FISHER SCIENTIFIC INC,2023,1st Quarter - Amendment,2023-06-26T14:20:08-04:00
5508,THERMO FISHER SCIENTIFIC INC,2023,2nd Quarter - Amendment,2023-06-26T19:39:22-04:00
5509,THERMO FISHER SCIENTIFIC INC,2023,2nd Quarter - Amendment,2023-06-26T19:44:25-04:00
5477,YAKIMA COUNTY,2022,4th Quarter - Report,2023-01-22T15:54:35-05:00


In [28]:
# View Total year for identified dups

df_clean[(df_clean['client_name'] == 'AEROVIRONMENT INC.') & (df_clean['filing_year'] == 2009)]

,source,query_company,role,filing_uuid,filing_year,period,filing_type,dt_posted,registrant_name,registrant_id,client_name,client_id,income,expenses,lobbying_issues,lobbying_bills,lobbying_gov_entities,url


In [29]:
# Creating a new column for base_quarter, and removing dups based on dt_posted datetime

df_clean['base_quarter'] = df_clean['filing_type'].str.replace(r'\s*-\s*(Report|Amendment).*', '', regex=True).str.strip()

df_deduped = (
    df_clean.sort_values('dt_posted', ascending=False)
            .drop_duplicates(subset=['client_name', 'filing_year', 'base_quarter'], keep='first')
            .reset_index(drop=True)
)


In [30]:
df_deduped[(df_deduped['client_name'] == 'AEROVIRONMENT INC.') & (df_deduped['filing_year'] == 2010)]

,source,query_company,role,filing_uuid,filing_year,period,filing_type,dt_posted,registrant_name,registrant_id,client_name,client_id,income,expenses,lobbying_issues,lobbying_bills,lobbying_gov_entities,url,base_quarter


In [31]:
# Sum the expenses for the deduped years

df_deduped.groupby('filing_year')['expenses'].sum().reset_index().style.format({'expenses': '${:,.0f}'})



,filing_year,expenses
0,1999,"$25,725,465"
1,2000,"$32,661,195"
2,2001,"$32,817,972"
3,2002,"$36,610,498"
4,2003,"$44,720,260"
5,2004,"$34,626,568"
6,2005,"$33,697,260"
7,2006,"$43,759,642"
8,2007,"$46,103,885"
9,2008,"$59,569,532"


In [32]:
df_deduped.groupby(['client_name', 'filing_year'])['expenses'].sum().reset_index().style.format({'expenses': '${:,.0f}'})


,client_name,filing_year,expenses
0,ACADEMY OF MANAGED CARE PHARMACY,2016,$0
1,ACADEMY OF MANAGED CARE PHARMACY,2017,$0
2,ACADEMY OF MANAGED CARE PHARMACY,2018,$0
3,ACADEMY OF MANAGED CARE PHARMACY,2019,$0
4,ACADEMY OF MANAGED CARE PHARMACY,2020,$0
5,ACADEMY OF MANAGED CARE PHARMACY,2021,$0
6,"ACCREDITATION ASSOCIATION FOR AMBULATORY HEALTH CARE, INC. (AAAHC)",2013,$0
7,"ACCREDITATION ASSOCIATION FOR AMBULATORY HEALTH CARE, INC. (AAAHC)",2014,$0
8,"ACCREDITATION ASSOCIATION FOR AMBULATORY HEALTH CARE, INC. (AAAHC)",2015,$0
9,"ACCREDITATION ASSOCIATION FOR AMBULATORY HEALTH CARE, INC. (AAAHC)",2016,$0


## Step 3 - Vizualizations 

In [33]:
# installing additional dependancies 

import sys
!{sys.executable} -m pip install --upgrade nbformat

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
# Graphing each companies summed expenses per year

df_yearly = df_deduped.groupby(['client_name', 'filing_year'])['expenses'].sum().reset_index()

fig = px.line(df_yearly, x='filing_year', y='expenses', color='client_name', markers=True,
              labels={'filing_year': 'Year', 'expenses': 'Lobbying Expenses', 'client_name': 'Company'})
fig.update_layout(hoverlabel=dict(bgcolor='white', font_color='black', bordercolor='lightgrey'))
fig.update_layout(title='Total Company Lobbying Expenses by Year')

fig.show()

fig.write_image('Viz/Company_change_expenses_by_year.svg', scale=2)



In [35]:
# creating the % change dataset

df_pct = (
    df_deduped.groupby(['client_name', 'filing_year'])['expenses'].sum()
    .reset_index()
    .sort_values(['client_name', 'filing_year'])
)

df_pct['pct_change'] = df_pct.groupby('client_name')['expenses'].pct_change() * 100
df_pct['pct_change'] = df_pct['pct_change'].fillna(0)


df_pct.head(20)


,client_name,filing_year,expenses,pct_change
0,ACADEMY OF MANAGED CARE PHARMACY,2016,0.0,0.0
1,ACADEMY OF MANAGED CARE PHARMACY,2017,0.0,0.0
2,ACADEMY OF MANAGED CARE PHARMACY,2018,0.0,0.0
3,ACADEMY OF MANAGED CARE PHARMACY,2019,0.0,0.0
4,ACADEMY OF MANAGED CARE PHARMACY,2020,0.0,0.0
5,ACADEMY OF MANAGED CARE PHARMACY,2021,0.0,0.0
6,ACCREDITATION ASSOCIATION FOR AMBULATORY HEALT...,2013,0.0,0.0
7,ACCREDITATION ASSOCIATION FOR AMBULATORY HEALT...,2014,0.0,0.0
8,ACCREDITATION ASSOCIATION FOR AMBULATORY HEALT...,2015,0.0,0.0
9,ACCREDITATION ASSOCIATION FOR AMBULATORY HEALT...,2016,0.0,0.0


In [42]:
# Graphing %change by company

fig = px.line(df_pct, x='filing_year', y='pct_change', color='client_name', markers=True,
              labels={'filing_year': 'Year', 'pct_change': '% Change in Expenses', 'client_name': 'Company'})
fig.add_hline(y=0, line_dash='dash', line_color='grey')
fig.update_layout(hoverlabel=dict(bgcolor='white', font_color='black', bordercolor='lightgrey'))
fig.update_layout(title='% Change in Lobbying Expenses by Year')

fig.write_image('Viz/%_change_expenses_by_year.svg', scale=2)

fig.show()


In [43]:
# showing totals

df_total = df_deduped.groupby('filing_year')['expenses'].sum().reset_index()
df_total['pct_change'] = df_total['expenses'].pct_change() * 100
df_total['pct_change'] = df_total['pct_change'].fillna(0)

fig1 = px.line(df_total, x='filing_year', y='expenses', markers=True,
               title='Total Lobbying Expenses by Year',
               labels={'filing_year': 'Year', 'expenses': 'Total Expenses'})
fig1.update_layout(hoverlabel=dict(bgcolor='white', font_color='black', bordercolor='lightgrey'))
fig1.show()

fig2 = px.line(df_total, x='filing_year', y='pct_change', markers=True,
               title='% Change in Total Lobbying Expenses by Year',
               labels={'filing_year': 'Year', 'pct_change': '% Change'})
fig2.add_hline(y=0, line_dash='dash', line_color='grey')
fig2.update_layout(hoverlabel=dict(bgcolor='white', font_color='black', bordercolor='lightgrey'))


fig1.write_image('Viz/total_expenses_by_year.svg', scale=2)
fig2.write_image('Viz/pct_change_total_expenses.svg', scale=2)

fig2.show()

In [44]:
df.head()

,source,query_company,role,filing_uuid,filing_year,period,filing_type,dt_posted,registrant_name,registrant_id,client_name,client_id,income,expenses,lobbying_issues,lobbying_bills,lobbying_gov_entities,url
0,Senate LDA,Lockheed Martin,client,49117f1a-a236-4583-a60e-919cc59e79c4,1999,NaN,Year-End Report,2000-02-14T00:00:00-05:00,THE RHOADS GROUP,6644,LOCKHEED MARTIN CORP,108908,120000.0,NaN,Defense; Telecommunications,NaN,HOUSE OF REPRESENTATIVES; SENATE,https://lda.senate.gov/filings/public/filing/4...
1,Senate LDA,Lockheed Martin,client,31f28e72-7236-4837-b0b6-8d262d1d6cd8,1999,NaN,Year-End Report,2000-02-14T00:00:00-05:00,VAN SCOYOC ASSOCIATES,39837,LOCKHEED MARTIN MISSILES & SPACE CO,147700,20000.0,NaN,Defense; Budget/Appropriations,NaN,HOUSE OF REPRESENTATIVES; SENATE,https://lda.senate.gov/filings/public/filing/3...
2,Senate LDA,Lockheed Martin,client,5d60eeb6-241e-4977-b0d1-852a36436cbf,1999,NaN,Mid-Year Termination (No Activity),1999-08-13T00:00:00-04:00,"CARTER GROUP, LLC",49008,LOCKHEED MARTIN AIRCRAFT & LOGISTICS CENTERS,156072,NaN,NaN,NaN,NaN,NaN,https://lda.senate.gov/filings/public/filing/5...
3,Senate LDA,Lockheed Martin,client,d52d60e0-0536-4207-8d87-9c1084e31025,1999,NaN,Year-End Report,2000-02-14T00:00:00-05:00,O'MELVENY & MYERS LLP,29904,LOCKHEED MARTIN CORP,135265,100000.0,NaN,Aerospace,NaN,HOUSE OF REPRESENTATIVES; SENATE,https://lda.senate.gov/filings/public/filing/d...
4,Senate LDA,Lockheed Martin,client,4e6ca9fd-7bad-451e-8a13-d7a698af4815,1999,NaN,Year-End Report,2000-02-14T00:00:00-05:00,DAP & ASSOC,48986,LOCKHEED MARTIN CORP,156067,120000.0,NaN,Telecommunications; Communications/Broadcastin...,NaN,HOUSE OF REPRESENTATIVES; SENATE,https://lda.senate.gov/filings/public/filing/4...


In [ ]:
# Top 15 government entities lobbied per company

df_entities = (
    df_deduped[df_deduped['lobbying_gov_entities'].notna() & (df_deduped['lobbying_gov_entities'] != '')]
    .copy()
)
df_entities['lobbying_gov_entities'] = df_entities['lobbying_gov_entities'].str.split('; ')
df_entities = df_entities.explode('lobbying_gov_entities')
df_entities['lobbying_gov_entities'] = df_entities['lobbying_gov_entities'].str.strip()
df_entities = df_entities[df_entities['lobbying_gov_entities'] != '']

entity_counts = (
    df_entities.groupby(['client_name', 'lobbying_gov_entities'])
    .size()
    .reset_index(name='filing_count')
)

top_entities = (
    entity_counts.sort_values('filing_count', ascending=False)
    .groupby('client_name', group_keys=False)
    .head(15)
    .reset_index(drop=True)
)

n_companies = top_entities['client_name'].nunique()

fig = px.bar(
    top_entities,
    x='filing_count',
    y='lobbying_gov_entities',
    facet_col='client_name',
    facet_col_wrap=1,
    orientation='h',
    labels={
        'filing_count': 'Number of Filings',
        'lobbying_gov_entities': 'Government Entity',
        'client_name': 'Company',
    },
    title='Top 15 Government Entities Lobbied by Company',
    height=450 * n_companies,
)
fig.update_layout(
    hoverlabel=dict(bgcolor='white', font_color='black', bordercolor='lightgrey'),
)
fig.update_yaxes(matches=None, showticklabels=True)
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))

fig.show()
fig.write_image('Viz/top_entities_by_company.svg', scale=2)

In [ ]:
# Government entities over time — filings & attributed expenses

# Explode entities (reuse same logic)
df_et = (
    df_deduped[df_deduped['lobbying_gov_entities'].notna() & (df_deduped['lobbying_gov_entities'] != '')]
    .copy()
)
df_et['lobbying_gov_entities'] = df_et['lobbying_gov_entities'].str.split('; ')
df_et = df_et.explode('lobbying_gov_entities')
df_et['lobbying_gov_entities'] = df_et['lobbying_gov_entities'].str.strip()
df_et = df_et[df_et['lobbying_gov_entities'] != '']

# Divide each filing's expenses evenly across its entities so totals aren't inflated
df_et['entity_count'] = df_et.groupby('filing_uuid')['lobbying_gov_entities'].transform('count')
df_et['attributed_expenses'] = df_et['expenses'] / df_et['entity_count']

# Limit to top 10 entities by total filing appearances
top10 = (
    df_et.groupby('lobbying_gov_entities').size()
    .nlargest(10).index
)
df_et = df_et[df_et['lobbying_gov_entities'].isin(top10)]

# ── Graph 1: filing count per entity per year ─────────────────────────────────

entity_year_count = (
    df_et.groupby(['filing_year', 'lobbying_gov_entities'])
    .size()
    .reset_index(name='filing_count')
)

fig1 = px.area(
    entity_year_count,
    x='filing_year',
    y='filing_count',
    color='lobbying_gov_entities',
    title='Government Entities Lobbied Over Time (Top 10)',
    labels={
        'filing_year': 'Year',
        'filing_count': 'Number of Filings',
        'lobbying_gov_entities': 'Government Entity',
    },
    height=550,
)
fig1.update_layout(hoverlabel=dict(bgcolor='white', font_color='black', bordercolor='lightgrey'))
fig1.show()
fig1.write_image('Viz/entities_filings_over_time.svg', scale=2)

# ── Graph 2: attributed expenses per entity per year ─────────────────────────

entity_year_exp = (
    df_et.groupby(['filing_year', 'lobbying_gov_entities'])['attributed_expenses']
    .sum()
    .reset_index()
)

fig2 = px.area(
    entity_year_exp,
    x='filing_year',
    y='attributed_expenses',
    color='lobbying_gov_entities',
    title='Lobbying Spend Attributed to Government Entities Over Time (Top 10)',
    labels={
        'filing_year': 'Year',
        'attributed_expenses': 'Attributed Expenses ($)',
        'lobbying_gov_entities': 'Government Entity',
    },
    height=550,
)
fig2.update_layout(hoverlabel=dict(bgcolor='white', font_color='black', bordercolor='lightgrey'))
fig2.show()
fig2.write_image('Viz/entities_expenses_over_time.svg', scale=2)

In [ ]:
# Stacked spend by government entity over time, faceted by company
# (each panel gets its own legend and x-axis labels)

from plotly.subplots import make_subplots
import plotly.graph_objects as go

df_ce = (
    df_deduped[df_deduped['lobbying_gov_entities'].notna() & (df_deduped['lobbying_gov_entities'] != '')]
    .copy()
)
df_ce['lobbying_gov_entities'] = df_ce['lobbying_gov_entities'].str.split('; ')
df_ce = df_ce.explode('lobbying_gov_entities')
df_ce['lobbying_gov_entities'] = df_ce['lobbying_gov_entities'].str.strip()
df_ce = df_ce[df_ce['lobbying_gov_entities'] != '']

df_ce['entity_count'] = df_ce.groupby('filing_uuid')['lobbying_gov_entities'].transform('count')
df_ce['attributed_expenses'] = df_ce['expenses'] / df_ce['entity_count']

# Top 10 entities globally by total attributed spend
top10 = (
    df_ce.groupby('lobbying_gov_entities')['attributed_expenses']
    .sum().nlargest(10).index.tolist()
)
df_ce = df_ce[df_ce['lobbying_gov_entities'].isin(top10)]

company_entity_year = (
    df_ce.groupby(['client_name', 'filing_year', 'lobbying_gov_entities'])['attributed_expenses']
    .sum().reset_index()
)

companies = sorted(company_entity_year['client_name'].unique())
colors    = px.colors.qualitative.Plotly
entity_color = {e: colors[i % len(colors)] for i, e in enumerate(top10)}
n = len(companies)

# vertical_spacing as a fraction of total height
v_spacing = 0.06
subplot_h = (1 - (n - 1) * v_spacing) / n

fig = make_subplots(
    rows=n, cols=1,
    subplot_titles=companies,
    shared_xaxes=False,
    vertical_spacing=v_spacing,
)

for i, company in enumerate(companies):
    df_c = (
        company_entity_year[company_entity_year['client_name'] == company]
        .sort_values('filing_year')
    )
    legend_ref = 'legend' if i == 0 else f'legend{i + 1}'

    for entity in top10:
        df_e = df_c[df_c['lobbying_gov_entities'] == entity]
        if df_e.empty:
            continue
        fig.add_trace(
            go.Scatter(
                x=df_e['filing_year'],
                y=df_e['attributed_expenses'],
                name=entity,
                mode='lines',
                stackgroup=f'stack_{i}',
                fillcolor=entity_color[entity],
                line=dict(color=entity_color[entity], width=0.5),
                showlegend=True,
                legend=legend_ref,
            ),
            row=i + 1, col=1,
        )

    # Position this panel's legend to the right of its subplot
    center_y = 1 - i * (subplot_h + v_spacing) - subplot_h / 2
    legend_cfg = dict(
        x=1.01, y=center_y, yanchor='middle', xanchor='left',
        bgcolor='rgba(255,255,255,0.85)', bordercolor='lightgrey',
        borderwidth=1, font=dict(size=9),
    )
    if i == 0:
        fig.update_layout(legend=legend_cfg)
    else:
        fig.update_layout(**{f'legend{i + 1}': legend_cfg})

fig.update_xaxes(showticklabels=True, title_text='Year')
fig.update_yaxes(title_text='Attributed Expenses ($)')
fig.update_layout(
    title='Lobbying Spend by Government Entity Over Time — by Company (Top 10 Entities)',
    height=420 * n,
    hoverlabel=dict(bgcolor='white', font_color='black', bordercolor='lightgrey'),
)

fig.show()
fig.write_image('Viz/company_entity_spend_over_time.svg', scale=2)